In [1]:
##### Aligns sub-national capital stock data with national total
# defines functions for aligning data across each country
# runs functions and combines into final dataset

# uses shares from sub-regions from whatever year they are available (closest to 2020)
# but applies to national total in 2020 from FAO (assumes sub-regional distribution is the same)

import os
import pandas as pd

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
FAO_capital = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/FAO_capital_stock_adjusted.csv")

# Set save path
save_path = f"{cd}/Data/Clean/Capital_stock/subnational_capital_stock_final.csv"

In [3]:
### Define function for aligning with national total 

def align_to_national_USD (country_code):
    sub_capital = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/{country_code}_capital_stock.csv")

    # pull national total based on ISO3
    country_labor = FAO_capital[FAO_capital['ISO3'] == country_code]

    # merge to sub_national labor
    sub_capital = sub_capital.merge(country_labor, on='ISO3', how='left')
    sub_capital['national_ag_capital_stock_USD_nominal'] = sub_capital['ag_capital_stock_mil_USD_nominal'] * 1e6

    # calculate regional shares
    region_sum = sub_capital['ag_capital_stock_2020_USD'].sum()
    sub_capital['region_sum'] = region_sum

    sub_capital['regional_share'] = sub_capital['ag_capital_stock_2020_USD'] / sub_capital['region_sum']

    # apportion jobs 
    sub_capital['ag_capital_stock_USD_nominal'] = sub_capital['national_ag_capital_stock_USD_nominal'] * sub_capital['regional_share']

    # add project ID
    sub_capital['PROJ_ID'] = sub_capital['ISO3'].astype(str) + '_' + sub_capital['GEO_ID'].astype(str)

    # select final column order 
    col_to_keep = ['ISO3', 'GEO_ID', 'PROJ_ID', 'GEO_ID_NAME', 'ag_capital_stock_USD_nominal']
    sub_capital = sub_capital[col_to_keep]

    return sub_capital

def align_to_national_count (country_code):
    sub_capital = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/{country_code}_capital_stock.csv")

    # pull national total based on ISO3
    country_labor = FAO_capital[FAO_capital['ISO3'] == country_code]

    # merge to sub_national labor
    sub_capital = sub_capital.merge(country_labor, on='ISO3', how='left')
    sub_capital['national_ag_capital_stock_USD_nominal'] = sub_capital['ag_capital_stock_mil_USD_nominal'] * 1e6

    # calculate regional shares
    region_sum = sub_capital['ag_capital_stock_count_machinery'].sum()
    sub_capital['region_sum'] = region_sum

    sub_capital['regional_share'] = sub_capital['ag_capital_stock_count_machinery'] / sub_capital['region_sum']

    # apportion jobs 
    sub_capital['ag_capital_stock_USD_nominal'] = sub_capital['national_ag_capital_stock_USD_nominal'] * sub_capital['regional_share']

    # add project ID
    sub_capital['PROJ_ID'] = sub_capital['ISO3'].astype(str) + '_' + sub_capital['GEO_ID'].astype(str)

    # select final column order 
    col_to_keep = ['ISO3', 'GEO_ID', 'PROJ_ID', 'GEO_ID_NAME', 'ag_capital_stock_USD_nominal']
    sub_capital = sub_capital[col_to_keep]

    return sub_capital

In [4]:
#### Run US seperate because of country code issue

US_capital = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/USA_capital_stock.csv")

# pull national total based on ISO3
country_labor = FAO_capital[FAO_capital['ISO3'] == 'USA']

# merge to sub_national labor
US_capital = US_capital.merge(country_labor, on='ISO3', how='left')
US_capital['national_ag_capital_stock_USD_nominal'] = US_capital['ag_capital_stock_mil_USD_nominal'] * 1e6

# calculate regional shares
US_sum = US_capital['ag_capital_stock_2020_USD'].sum()
US_capital['region_sum'] = US_sum

US_capital['regional_share'] = US_capital['ag_capital_stock_2020_USD'] / US_capital['region_sum']

# apportion jobs 
US_capital['ag_capital_stock_USD_nominal'] = US_capital['national_ag_capital_stock_USD_nominal'] * US_capital['regional_share']

# add project ID
US_capital["GEO_ID"] = US_capital["GEO_ID"].astype(str).str.zfill(5)
US_capital['PROJ_ID'] = US_capital['ISO3'].astype(str) + '_' + US_capital['GEO_ID'].astype(str)

# select final column order 
col_to_keep = ['ISO3', 'PROJ_ID', 'GEO_ID', 'GEO_ID_NAME', 'ag_capital_stock_USD_nominal']
US_capital = US_capital[col_to_keep]

In [5]:
### Run alignment for all available countries

country_codes_USD = ['BEL', 'BGR', 'CAN', 'CHN', 'DEU', 'ESP', 'FIN', 'FRA', 'GBR', 'GRC', 'HRV', 'HUN',
                 'ITA', 'MEX', 'POL', 'PRT', 'ROU',
                 'SWE']

country_codes_count = ['BRA', 'EGY', 'GHA', 'IND', 'TUR', 'ZAF']

aligned_USD = {}
for code in country_codes_USD:
    aligned_USD[code] = align_to_national_USD(code)

aligned_count = {}
for code in country_codes_count:
    aligned_count[code] = align_to_national_count(code)

# concatinate all countries 
subnational_capital_USD = pd.concat(aligned_USD, ignore_index=True)
subnational_capital_count = pd.concat(aligned_count, ignore_index=True)
subnational_capital = pd.concat([subnational_capital_USD, subnational_capital_count, US_capital], ignore_index=True)

subnational_capital = subnational_capital.dropna()

In [6]:
# Save
subnational_capital.to_csv(save_path, index=False)